In [ ]:
using Plots
include("coevolution_network_base.jl")
using .CoevolutionNetworkBase
using Printf
theme(:dracula)

In [ ]:
# Parameters
L = 40.0
dx = 0.05
x = -L/2:dx:L/2-dx
r = 3.0
M = 15
beta = 2.5
alpha = 0.0
gamma = 1.0
# D = 0.0025
D = 0.01
Nh = 2 * 10^6
stochastic = true
sigma = 1.0

# Initialize viral and immune densities
viral_density = zeros(Float64, length(x))
viral_density[Int(round(length(x)/2))] = 100/dx

# initial_variance = 0.3
# viral_density .= 100/sqrt(2 * pi * initial_variance) .* exp.(-x.^2/2/initial_variance)
viral_density2 = zeros(Float64, length(x))
immune_density = zeros(Float64, length(x))

# Create Population instances
population = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density, immune_density;stochastic=stochastic, sigma=sigma)
population2 = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density2, immune_density; stochastic=stochastic, sigma=sigma)
populations = [population, population2]

# populations[1] = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density, immune_density;stochastic=true)

# Create Network instance
migration_matrix = 1e-3 * ones(size(populations,1),size(populations,1)) # Define an appropriate migration matrix
println(size(migration_matrix,1))
network = Network(populations, migration_matrix);

# Create Simulation instance
dt = 0.05 # Define an appropriate time step size
duration = 80.0 # Define an appropriate simulation duration
simulation = Simulation(network, dt, duration; thin_by=1);

@time run_simulation!(simulation);

total_infected_per_deme = calculate_total_infected_per_deme(simulation)
println(total_infected_per_deme[1,end] > 0)

In [ ]:
variances_per_deme = calculate_antigenic_variance_per_deme(simulation)
total_infected_per_deme = calculate_total_infected_per_deme(simulation)

# Number of demes
num_demes = size(total_infected_per_deme, 1)

# Plot for Variance of Antigenicity vs Time
treg = simulation.duration_times .< 20
p = plot(;xlabel="Time", ylabel="Variance of antigenicity", background_color=:black)
p2 = plot(xlabel="Time", ylabel="Total Infected", background_color=:black, yscale=:log, legend=:best)
p3 = plot(xlabel="Total Infected", ylabel="Variance of antigenicity", xscale=:log10, legend=:topleft, background_color=:black)


for deme in 1:num_demes
    plot!(p, simulation.duration_times[treg], variances_per_deme[deme, treg], label="Deme $deme")
end
plot!(p, simulation.duration_times[treg], 0 .+ 2 * D * simulation.duration_times[treg], linestyle=:dash, label=:none)
plot!(p, simulation.duration_times[treg] .+ 4.3, 2 * D * simulation.duration_times[treg], linestyle=:dash, label=:none)
plot!(p, ylim=(0,2))
display(p)

# Plot for Total Infected vs Time
treg = simulation.duration_times .< 90

# Manually setting y-axis ticks for each order of magnitude
# yticks = [10^i for i in 0:7]; plot!(p2, yticks=yticks)
for deme in 1:num_demes
    plot!(p2, simulation.duration_times[treg], total_infected_per_deme[deme, treg] .+ 10^-15, label="Deme $deme")
end
# plot!(p2, simulation.duration_times[treg] .- 4.3, total_infected_per_deme[2, treg] .+ 10^-15, label="Deme 2, shifted", xlim=(0,Inf))
display(p2)

for deme in 1:num_demes
    # Find the index where the total infected per deme attains its maximum for the first time
    max_index = findfirst(==(maximum(total_infected_per_deme[deme, :])), total_infected_per_deme[deme, :])

    # Create the Boolean mask
    # reg = (total_infected_per_deme[deme, :] .> 100) .& (variances_per_deme[deme, :] .> 0) .& (1:length(total_infected_per_deme[deme, :]) .< max_index)
    reg = (total_infected_per_deme[deme, :] .> 0) .& (simulation.duration_times .<  simulation.duration_times[argmax(total_infected_per_deme[deme, 1:50])])
    reg = (total_infected_per_deme[deme, :] .> 0) .& (simulation.duration_times .<  20)
    # Plot using the mask
    plot!(p3, total_infected_per_deme[deme, reg], variances_per_deme[deme, reg], label="Deme $deme")
end
display(p3)

In [ ]:
# Parameters
L = 40.0
dx = 0.05
x = -L/2:dx:L/2-dx
r = 3.0
M = 15
beta = 2.5
alpha = 0.0
gamma = 1.0
# D = 0.0025
D = 0.01
Nh = 2 * 10^6
stochastic = true
sigma = 1.0

# Initialize viral and immune densities
viral_density = zeros(Float64, length(x))
viral_density[Int(round(length(x)/2))] = 100/dx

# initial_variance = 0.3
# viral_density .= 100/sqrt(2 * pi * initial_variance) .* exp.(-x.^2/2/initial_variance)
viral_density2 = zeros(Float64, length(x))
immune_density = zeros(Float64, length(x))

# Create Population instances
population = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density, immune_density;stochastic=stochastic, sigma=sigma)
population2 = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density2, immune_density; stochastic=stochastic, sigma=sigma)
populations = [population, population2]

# populations[1] = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density, immune_density;stochastic=true)

# Create Network instance
migration_matrix = 1e-5 * ones(size(populations,1),size(populations,1)) # Define an appropriate migration matrix
println(size(migration_matrix,1))
network = Network(populations, migration_matrix);

# Create Simulation instance
dt = 0.05 # Define an appropriate time step size
duration = 80.0 # Define an appropriate simulation duration
simulation = Simulation(network, dt, duration; thin_by=1);

@time run_simulation!(simulation);

total_infected_per_deme = calculate_total_infected_per_deme(simulation)
println(total_infected_per_deme[1,end] > 0)
F_ST = calculate_FST(simulation)
plot(simulation.duration_times, F_ST, xlims=(0, 20))

In [ ]:
using Plots
using LaTeXStrings

migration_rates = [1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0]  # adjust as needed
colors = distinguishable_colors(length(migration_rates))

FSTs = []

for (i, mig_rate) in enumerate(migration_rates)
    # Reset viral and immune densities
    viral_density = zeros(length(x))
    viral_density[Int(round(length(x)/2))] = 100/dx
    viral_density2 = zeros(length(x))
    immune_density = zeros(length(x))

    # Create populations and network
    pop1 = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density, immune_density; stochastic=stochastic, sigma=sigma)
    pop2 = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density2, immune_density; stochastic=stochastic, sigma=sigma)
    pops = [pop1, pop2]

    mig_matrix = mig_rate * ones(2, 2)
    network = Network(pops, mig_matrix)

    sim = Simulation(network, dt, duration; thin_by=1)
    run_simulation!(sim)

    push!(FSTs, (sim.duration_times, calculate_FST(sim)))
end

# Plotting
plot(ylims=(0.0, 1.5))
for (i, (ts, fst)) in enumerate(FSTs)
    plot!(ts, fst, label = "m = $(migration_rates[i])", lw=2, color=colors[i])
end
xlabel!(L"time")
ylabel!(L"F_{ST}")
title!("F_{ST} over time for different migration rates")
xlims!(0, 20)


In [ ]:

# Plotting
plot(ylims=(0.0, 1.5))
for (i, (ts, fst)) in enumerate(FSTs[1:2])
    plot!(ts, fst, label = "m = $(migration_rates[i])", lw=2, color=colors[i])
end
xlabel!(L"time")
ylabel!(L"F_{ST}")
title!("F_{ST} over time for different migration rates")
xlims!(0, 20)


In [ ]:
using Plots
using LaTeXStrings
using Statistics

migration_rates = [1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0]
colors = distinguishable_colors(length(migration_rates))
n_replicates = 50

FSTs_mean = []
FSTs_std = []

for (i, mig_rate) in enumerate(migration_rates)
    FST_reps = []

    for rep in 1:n_replicates
        println(rep)
        # Reset viral and immune densities
        viral_density = zeros(length(x))
        viral_density[Int(round(length(x)/2))] = 100/dx
        viral_density2 = zeros(length(x))
        immune_density = zeros(length(x))

        # Create populations and network
        pop1 = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density, immune_density; stochastic=stochastic, sigma=sigma)
        pop2 = Population(L, dx, r, M, beta, alpha, gamma, D, Nh, viral_density2, immune_density; stochastic=stochastic, sigma=sigma)
        pops = [pop1, pop2]

        mig_matrix = mig_rate * ones(2, 2)
        network = Network(pops, mig_matrix)

        sim = Simulation(network, dt, 20.0; thin_by=1)
        run_simulation!(sim)

        push!(FST_reps, calculate_FST(sim))
    end

    FST_mat = hcat(FST_reps...)  # [time × replicate]
    push!(FSTs_mean, mean(FST_mat, dims=2)[:])
    push!(FSTs_std, std(FST_mat, dims=2)[:])
end

# Plotting
plot(ylims=(0.0, 1.0))
for (i, μ) in enumerate(FSTs_mean)
    ts = simulation.duration_times
    σ = FSTs_std[i]
    plot!(ts, μ, ribbon=σ, label="m = $(migration_rates[i])", lw=2, color=colors[i])
end

xlabel!(L"time")
ylabel!(L"F_{ST}")
title!("Mean ± SD of F_{ST}(t) for different migration rates")
xlims!(0, 20)


In [ ]:
# Plotting
p = plot(ylims=(0.0, 1.5), legend=false)
ts = simulation.duration_times
for (i, μ) in enumerate(FSTs_mean)
    σ = FSTs_std[i]
    plot!(p, ts, μ, ribbon=σ, label="m = $(migration_rates[i])", lw=2, color=colors[i])
end

xlabel!(p, L"time")
ylabel!(p, L"F_{ST}")
title!(p, "Mean ± SD of F_{ST}(t) for different migration rates")
display(p)